# omnia-sdk — the GPU benchmark

This produces the artefact the README calls auditable: a JSON file recording what
this machine is, what it measured, and how. Run it top to bottom on a Colab GPU
runtime and download the result at the end.

**Why it must be a GPU runtime.** The claim omnia-sdk makes is that it removes idle
GPU time. On a CPU there is no GPU to leave idle, so utilisation cannot be sampled
and the end-to-end speedup cannot be measured — the data-feeding number is still
real, but it is only half the story and the file will say so.

`Runtime → Change runtime type → T4 GPU` before you start.


## 1. Confirm there is a GPU


In [ ]:
import subprocess, sys

out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True)
if out.returncode != 0:
    raise SystemExit(
        'No GPU on this runtime.\n'
        'Runtime -> Change runtime type -> T4 GPU, then run this cell again.\n'
        'Running without one produces a file that cannot support the README figures.')
print(out.stdout.strip())


## 2. Install omnia-sdk from source

Installed straight from the public repository, so what runs here is what a reader
gets — no uploaded folder that might differ from the published code.


In [ ]:
%%bash
pip install -q 'git+https://github.com/mishel-0/omnia-sdk.git#egg=omnia-sdk[svs]' 2>&1 | tail -2
pip install -q openslide-python pillow 2>&1 | tail -1
apt-get -qq install -y openslide-tools > /dev/null 2>&1 || true
python -c "import omnia_sdk; print('omnia-sdk', omnia_sdk.__version__)"


## 3. Confirm the package is honest before trusting its timings

The test suite checks that a tile survives the container unchanged. If that fails,
no speed number below is worth reading — a faster loader that alters pixels is not
a faster loader.


In [ ]:
%%bash
pip install -q pytest
git clone -q https://github.com/mishel-0/omnia-sdk.git /content/omnia-sdk-src 2>/dev/null || true
cd /content/omnia-sdk-src && python -m pytest tests/ -q 2>&1 | tail -3


## 4. Run the benchmark

This is `python -m omnia_sdk.benchmark` — the same command in the README, run
unmodified. It downloads a public Aperio test slide, converts it, and times both
paths over several epochs with the first dropped as warm-up.

Ten minutes or so on a T4.


In [ ]:
!python -m omnia_sdk.benchmark \
    --tiles 5000 --epochs 5 --batch 64 \
    --output /content/benchmark_results.json


## 5. Read what it recorded


In [ ]:
import json
r = json.load(open('/content/benchmark_results.json'))
m, meas = r['machine'], r['measurement']

print('MACHINE')
cpu = m['cpu'] if isinstance(m['cpu'], dict) else {'model': m['cpu']}
print(f"  cpu          {cpu.get('model')}  ({cpu.get('cores')} cores, {cpu.get('ram_gb')} GB)")
print(f"  gpu          {m['gpu']}")
print(f"  context      {m.get('measurement_context', '')}")
print()
print('MEASURED')
print(f"  data feeding   {meas['data_speedup']}x")
print(f"  full train     {meas.get('full_train')}")
print(f"  regime         {meas.get('regime')}")
print(f"  gpu util .svs   {meas.get('gpu_util_svs')}")
print(f"  gpu util .omnia {meas.get('gpu_util_omnia')}")

# The point of running on a GPU is these three being populated.
missing = [k for k in ('full_train', 'regime') if not meas.get(k)]
if not meas.get('gpu_util_svs'):
    missing.append('gpu_util_svs')
if missing:
    print()
    print('  INCOMPLETE — still missing:', ', '.join(missing))
    print('  This file cannot support the utilisation figures in the README.')
else:
    print()
    print('  Complete. This file supports the README figures.')


## 6. Download it

Commit the downloaded file to `benchmarks/benchmark_results.json`. It replaces the
CPU-only run, which recorded empty GPU-utilisation fields and could not back the
utilisation claims the README makes.


In [ ]:
from google.colab import files
files.download('/content/benchmark_results.json')


---

### Reading this honestly

**Data-loading speedup** is a property of the format and holds on any machine.
It is the largest number here and the least interesting, because it measures the
distance from openslide — the slowest common starting point, not the best
alternative. If you already run cuCIM your baseline is far higher.

**End-to-end speedup** depends on your model. A small model is data-bound and gains
most of the data-loading win; a large one is compute-bound and gains little. The
`regime` field says which of the two this run landed in, and the honest answer for
a big model is that this format will not help you much.

**GPU utilisation** is the figure that actually justifies the project: it is the
share of paid GPU time spent waiting for JPEG-2000 decoding rather than training.
It can only be sampled where a GPU exists, which is the entire reason this notebook
refuses to run without one.
